# Example: Validation Report and Compliance Config

In this example, we evaluate the Session 3 engine variants against four formal deployment gates, produce a **validation report** with pass/fail verdicts, and export the **compliance configuration** that Session 4's production system enforces in real time.

> **By the end of this example, you will be able to:**
> * __Apply the four validation gates:__ Evaluate each strategy (EWLS engine, bandit-learned sigma, frozen baseline, buy-and-hold) against Sharpe, drawdown, failure rate, and benchmark thresholds. Produce a pass/fail table.
> * __Export compliance configuration:__ Define pre-approved operating limits (concentration cap, drawdown gate, turnover limit, position size limit) and save them for Session 4.
> * __Produce a strategy comparison dashboard:__ Visualize the distributional performance of all strategies side by side using box plots and histograms.

This is the gate between simulation and production.

___

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

### Constants


In [ ]:
# Validation inputs
EWLS_RESULTS_PATH = joinpath(_PATH_TO_DATA, "ewls-replay-results.jld2")
SIGMA_BANDIT_RESULTS_PATH = joinpath(_PATH_TO_DATA, "sigma-bandit-results.jld2")
B₀ = 10_000.0
VALIDATION_MIN_SHARPE = 0.3
VALIDATION_MAX_DRAWDOWN = 0.25
VALIDATION_MAX_FAILURE_RATE = 0.10


Load the backtest results from Examples 1 and 2. The EWLS replay results provide frozen and online engine distributions. The sigma-bandit results provide bandit-learned and heuristic CES distributions.

In [ ]:
frozen_fw, frozen_dd, frozen_sharpe, ewls_fw, ewls_dd, ewls_sharpe, n_paths, bandit_fw, bandit_dd, bandit_sharpe, heur_fw, heur_dd, heur_sharpe, B\u2080 = let
    # --- Step 1: Load EWLS results from Example 1 ---
    ewls_data = load_results(EWLS_RESULTS_PATH);
    frozen_fw     = ewls_data["frozen_final_wealth"];
    frozen_dd     = ewls_data["frozen_max_dd"];
    frozen_sharpe = ewls_data["frozen_sharpe"];
    ewls_fw       = ewls_data["ewls_final_wealth"];
    ewls_dd       = ewls_data["ewls_max_dd"];
    ewls_sharpe   = ewls_data["ewls_sharpe"];
    n_paths       = ewls_data["n_paths"];

    # --- Step 2: Load sigma-bandit results from Example 2 ---
    bandit_data = load_results(SIGMA_BANDIT_RESULTS_PATH);
    bandit_fw     = bandit_data["bandit_final_wealth"];
    bandit_dd     = bandit_data["bandit_max_dd"];
    bandit_sharpe = bandit_data["bandit_sharpe"];
    heur_fw       = bandit_data["heur_final_wealth"];
    heur_dd       = bandit_data["heur_max_dd"];
    heur_sharpe   = bandit_data["heur_sharpe"];

    B\u2080 = 10_000.0;
    println("Loaded results: $(n_paths) paths, 4 strategies")
    return frozen_fw, frozen_dd, frozen_sharpe, ewls_fw, ewls_dd, ewls_sharpe, n_paths, bandit_fw, bandit_dd, bandit_sharpe, heur_fw, heur_dd, heur_sharpe, B\u2080
end


___
## Task 1: Apply the Four Validation Gates
We evaluate each strategy against the deployment criteria: median Sharpe $\geq$ 0.3, median max drawdown $\leq$ 25%, failure rate $\leq$ 10%, and beats buy-and-hold on median wealth.

> __What should you see?__
>
> The EWLS and bandit-sigma strategies should pass most or all gates. The frozen baseline may fail on Sharpe or drawdown if it cannot adapt to regime shifts. Buy-and-hold is the benchmark, not a candidate.

In [ ]:
reports = let
    # --- Step 1: Define validation criteria ---
    bh_median_wealth = median(frozen_fw);  # use frozen as conservative buy-and-hold proxy

    criteria = Dict(
        "min_sharpe" => VALIDATION_MIN_SHARPE,
        "max_drawdown" => VALIDATION_MAX_DRAWDOWN,
        "max_failure_rate" => VALIDATION_MAX_FAILURE_RATE,
        "min_wealth_vs_bh" => bh_median_wealth
    );

    # --- Step 2: Build validation reports ---
    strategies = [
        ("EWLS Engine (h=63)", ewls_fw, ewls_dd, ewls_sharpe),
        ("Bandit-\u03c3 CES", bandit_fw, bandit_dd, bandit_sharpe),
        ("Heuristic-\u03c3 CES", heur_fw, heur_dd, heur_sharpe),
        ("Frozen CD Engine", frozen_fw, frozen_dd, frozen_sharpe),
    ];

    reports = MyValidationReport[];
    for (label, fw, dd, sr) in strategies
        actuals = Dict(
            "min_sharpe" => median(sr),
            "max_drawdown" => median(dd),
            "max_failure_rate" => mean(fw .< B\u2080),
            "min_wealth_vs_bh" => median(fw)
        );
        report = build(MyValidationReport;
            strategy_label = label, criteria = criteria, actuals = actuals);
        push!(reports, report);
    end

    # --- Step 3: Display pass/fail table ---
    println("="^80)
    println("  VALIDATION REPORT: Pass/Fail Deployment Gates")
    println("="^80)
    for r in reports
        all_pass = all(values(r.passed));
        verdict = all_pass ? "PASS" : "FAIL";
        println("\n  Strategy: $(r.strategy_label)  [$(verdict)]")
        for (k, v) in r.passed
            actual = round(r.actuals[k], digits=3);
            threshold = round(r.criteria[k], digits=3);
            status = v ? "\u2713" : "\u2717";
            println("    $(status) $(k): actual=$(actual), threshold=$(threshold)")
        end
    end
    return reports
end


___
## Task 2: Export Compliance Configuration
We define the pre-approved operating limits for Session 4's production system. These limits determine which trades auto-execute and which queue for human review.

> __What should you see?__
>
> A dictionary of compliance parameters saved to disk. These values come from the validation analysis: the drawdown gate matches the engine's trigger rule, the concentration cap limits single-asset exposure, and the turnover limit bounds daily trading activity.

In [ ]:
compliance_config = let
    # --- Step 1: Build compliance config ---
    compliance_config = build_compliance_config(;
        concentration_cap = 0.40,
        drawdown_gate = 0.15,
        turnover_limit = 0.50,
        position_size_limit = 5000.0
    );

    # --- Step 2: Display ---
    println("="^60)
    println("  COMPLIANCE CONFIGURATION FOR SESSION 4")
    println("="^60)
    for (k, v) in sort(collect(compliance_config))
        println("  $(k): $(v)")
    end
    println("\nTrades within these limits auto-execute.")
    println("Trades exceeding any limit queue for human review.")

    # --- Step 3: Save for Session 4 ---
    save_results(joinpath(_PATH_TO_DATA, "compliance-config.jld2"), compliance_config);
    println("\nSaved to compliance-config.jld2")
    return compliance_config
end


___
## Task 3: Strategy Comparison Dashboard
We visualize the distributional performance of all strategies side by side. Box plots show the spread of terminal wealth and maximum drawdown; histograms show the Sharpe ratio distribution.

> __What should you see?__
>
> Strategies that passed validation should have higher median wealth, lower drawdown spread, and a right-shifted Sharpe distribution compared to those that failed.

In [ ]:
let
    # --- Step 1: Terminal wealth box plot ---
    labels = ["EWLS" "Bandit-\u03c3" "Heuristic-\u03c3" "Frozen"];
    wealth_data = hcat(ewls_fw ./ B\u2080, bandit_fw ./ B\u2080, heur_fw ./ B\u2080, frozen_fw ./ B\u2080);

    p1 = boxplot(labels, wealth_data, legend=false, ylabel="W/W\u2080",
        title="Terminal Wealth Distribution", color=:steelblue, alpha=0.7);
    hline!(p1, [1.0], linestyle=:dash, color=:red, label="");

    # --- Step 2: Max drawdown box plot ---
    dd_data = hcat(ewls_dd .* 100, bandit_dd .* 100, heur_dd .* 100, frozen_dd .* 100);
    p2 = boxplot(labels, dd_data, legend=false, ylabel="Max Drawdown (%)",
        title="Drawdown Distribution", color=:coral, alpha=0.7);
    hline!(p2, [25.0], linestyle=:dash, color=:red, label="");

    # --- Step 3: Sharpe histogram overlay ---
    p3 = histogram(ewls_sharpe, bins=40, alpha=0.5, label="EWLS", color=:steelblue);
    histogram!(p3, bandit_sharpe, bins=40, alpha=0.5, label="Bandit-\u03c3", color=:goldenrod);
    histogram!(p3, frozen_sharpe, bins=40, alpha=0.3, label="Frozen", color=:gray50);
    vline!(p3, [0.3], linestyle=:dash, color=:red, label="Gate");
    xlabel!(p3, "Sharpe Ratio"); ylabel!(p3, "Count");
    title!(p3, "Sharpe Distribution");

    display(plot(p1, p2, p3, layout=(1,3), size=(1200, 400)))
end

___
## Summary
This example applied four formal validation gates to four strategies, produced pass/fail verdicts, exported a compliance configuration for Session 4, and visualized the distributional comparison. The compliance config is saved to `compliance-config.jld2`.

### Key Takeaways
* __Validation gates produce a binary deployment decision:__ Each strategy either passes all four criteria or fails. This replaces subjective judgment with an explicit, auditable process.
* __Compliance parameters bridge simulation to production:__ The concentration cap, drawdown gate, turnover limit, and position size limit become hard constraints in Session 4's execution system.
* __The dashboard makes distributional differences visible:__ Box plots and histograms reveal not just median performance but the full spread of outcomes, exposing tail risks that summary statistics hide.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.